# Chapter 1 · Notebook 2 of 3 — A Very Simple Tool Call

**AI Agent Course** · Nebius Token Factory × NVIDIA

This is the heart of the chapter: we build **our first controlled agentic loop**. The model will *request* an action, our Python code will *execute* it, and the result goes back to the model. Everything after this - real tools, multi-step agents - is this loop, but scaled up!

## Setup

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).

<details>
<summary>Show screenshot: Token Factory home</summary>

![Token Factory home](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/token-factory-home.png)

</details>

<details>
<summary>Show screenshot: API Key creation</summary>

![API Key creation](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/api-key-creation.png)

</details>

Your `lesson/.env` should look like:

```
NEBIUS_API_KEY=paste-your-key-here
```

**Why a `.env` file at all?** In real development, configuration that changes between people and machines (and *especially* secrets) lives outside your code. The `.env` file holds it; `load_dotenv()` reads it into environment variables. That way the same code runs on your laptop, your teammate's, and the server, and no key ever gets committed. **Key hygiene reminder:** keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it, unless you want someone stealing your Token Factory credits!

**Running in Google Colab?** Colab doesn't have your `.env`. Two options:

- Upload it: open the file browser (folder icon, left side) and drag your `.env` in, then point `load_dotenv()` at it with load_dotenv("path-to-your-env-file").
- Or use **Colab Secrets** (key icon, left side): add `NEBIUS_API_KEY` there, then run `from google.colab import userdata; os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")` instead of `load_dotenv()`.

In [ ]:
%pip install -q openai python-dotenv rich sympy

In [ ]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

# Colab users: comment the two lines above and use Colab Secrets instead:
# from google.colab import userdata
# os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

## 1. What is a tool call?

A *tool* is just a function; a *tool call* is just the model asking us to run it. In agents, this is how we execute actions. An agent could query a database, fetch some document, or call an API. Each of these are functions we can request our model to run.

## 2. How does the model see a tool?

Before the demo, it's worth uncovering the magic, because from the model's side there is **nothing special going on**:

1. **The tool's description is just text in the prompt.** When you register a function, its name, description, and parameters get placed into the model's context. The model "knows about the tool" the same way it knows about anything else you tell it.
2. **The JSON format is what the model was created to expect.** Tool descriptions are given as JSON schemas because models like Nemotron were *trained* on exactly this format. It's the shape of tool they're already comfortable reading!
3. **A tool call is just generated text.** The model doesn't *invoke* anything. It simply chooses to generate a string that happens to be JSON with a function name and arguments - and generating a string is the only thing a model ever does. Nothing suprising is happening!
4. **Because it's just text, it can go wrong.** the model might wrap the JSON in something unexpected, or produce a clumsy almost-JSON string. While there are ways we will cover to control tool-calling behavior, here we will build a naive version. You will see why it can be fragile.

Keep point 3 in mind for everything ahead: **the model never executes anything. Your code does.**

## 3. First contact: tool calling in the model card UI

Before writing any code:

1. Open the **Nemotron 3 Nano** model card playground and find the **Functions and JSON output** dropdown. Click **+ Add function**.

<details>
<summary>Show screenshot: Function dropdown</summary>

![Function dropdown](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card-tool-call.png)

</details>

2. Paste this function into the UI and click **Add function**.

```json
{
  "name": "get_weather",
  "description": "Get current weather for a city",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {"type": "string"}
    },
    "required": ["city"]
  }
}
```

<details>
<summary>Show screenshot: Tool call paste</summary>

![Tool call paste](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/tool-call-paste.png)

</details>

3. Ask **"What's the weather in Atlanta?"** and watch the model respond with a *tool call* instead of an answer.

Notice what the UI shows: the function name, the parsed arguments, and the fact that **nothing was executed.** The model only produced a *request* (point 3 above!). Someone (in a minute: our Python code) has to run the function and hand the result back.

<details>
<summary>Show screenshot: Tool call in the playground</summary>

![Tool call in the playground](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/tool-call-success.png)

</details>

Now that you've seen it, let's build one that can actually execute, from scratch.

## 4. Manual JSON tool call

When you ask an LLM to calculate a math equation, does it actually do computation? At the basic level, not really. It is familiar with mathematical concepts and expressions from its training, so it outputs information it *knows* rather than what it verifies by crunching numbers.

Our market-research assistant can't afford that - its numbers have to be exact. Let's create a **calculator tool** in Python that our LLM can actually use, so its answers are grounded in computation.

### Here is our tool, which uses SymPy to parse and evaluate math expressions.

In [ ]:
import sympy

def calculate(expression: str) -> str:
    return str(sympy.sympify(expression))

#### Now, we will use the tool we have created.

In [ ]:
import json

TOOL_SYSTEM_PROMPT = """You can use one tool:
calculate(expression: str) -> the exact result of an arithmetic expression
If the user's request needs the tool, respond with ONLY this JSON, no other text:
{"tool": "calculate", "arguments": {"expression": "<expression>"}}
Otherwise, answer normally."""


messages = [
    {"role": "system", "content": TOOL_SYSTEM_PROMPT},
    {"role": "user", "content": "If 4,823 stores each sell 3,917 units, how many units is that in total?"},
]

resp = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=messages,
    temperature=0,
)
raw = resp.choices[0].message.content
print("Model said:", raw)

call = json.loads(raw)
result = calculate(**call["arguments"])

# Feed the result back for a final answer
messages.append({"role": "assistant", "content": raw})
messages.append({"role": "user", "content": f"Tool result: {result}. Answer the user."})

final = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=messages,
)
print(final.choices[0].message.content)

**Walk the loop once more, slowly:** the model *generated a string* → our code parsed it and *chose* to execute → the result went back as a message → the model answered grounded in a real computation. That's the entire agentic loop!

### Exercise: Break it!

The code above works well, our tool call is simple and we can expect it to do as we asked. But what can go wrong here, especially when we have less view into what our model is and needs to do?

Your job: **make the loop crash (or misbehave) in at least three different ways**, using nothing but the user message.

Hints:
1. What should the model be able and not be able to do?
2. What if our inputs differ from what the function expects?
3. Inspect the tool itself - what can it do and what can't it do?

For each break you find, write one line: **what the model emitted → which line of our code died → whose fault it really was.**

In [ ]:
import json

TOOL_SYSTEM_PROMPT = """You can use one tool:
calculate(expression: str) -> the exact result of an arithmetic expression
If the user's request needs the tool, respond with ONLY this JSON, no other text:
{"tool": "calculate", "arguments": {"expression": "<expression>"}}
Otherwise, answer normally."""


messages = [
    {"role": "system", "content": TOOL_SYSTEM_PROMPT},
    {"role": "user", "content": ""}, ## What goes here?
]

resp = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=messages,
    temperature=0,
)
raw = resp.choices[0].message.content
print("Model said:", raw)

call = json.loads(raw)
result = calculate(**call["arguments"])

# Feed the result back for a final answer
messages.append({"role": "assistant", "content": raw})
messages.append({"role": "user", "content": f"Tool result: {result}. Answer the user."})

final = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=messages,
)
print(final.choices[0].message.content)

Notice the pattern in your breaks: the model did something *plausible* every time - it's our parsing that was brittle. The boundary between model output and your code is where agents fail, and hardening that boundary is a recurring theme from here on.

## Wrap-up → Notebook 3

The loop works. But *works* isn't the bar for something you'd ship: is it fast enough for a user to sit through? What does every trip around the loop cost? What happens when an agent takes ten trips per task? **Notebook 3** is where we put numbers on the loop!